# 1. Prompt Hierarchy and Composition Patterns

# A useful hierarchy is:
    System-level role and non-negotiable rules
    Task-specific instructions
    Context or reference data
    User input
    Output-format requirements

# Higher-level instructions should be stable. Dynamic user data should be inserted only into clearly marked placeholders.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Level 1: Stable system instructions
SYSTEM_PROMPT = """
You are a banking customer-support assistant.

Mandatory rules:
1. Use only the supplied account information.
2. Do not invent balances, transactions, or customer details.
3. Do not expose confidential information.
4. If the answer is unavailable, say:
   "The supplied information does not contain the answer."
"""

# Levels 2–5: Task, context, user input and output instructions
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),

    ("human", """
TASK:
Answer the customer's question.

ACCOUNT INFORMATION:
<account_data>
{account_data}
</account_data>

CUSTOMER QUESTION:
<question>
{question}
</question>

OUTPUT REQUIREMENTS:
- Use simple language.
- Maximum three sentences.
- Do not include information outside <account_data>.
""")
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "account_data": """
    Customer ID: C101
    Account type: Savings
    Available balance: ₹48,500
    Last transaction: ₹2,000 debit at ABC Store
    """,
    "question": "What is my balance and where was my last transaction?"
})

print(result)